In [3]:
import sys 
from pathlib import Path 
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from xgboost import XGBClassifier

In [4]:
root_dir = Path.cwd().parent
sys.path.append(str(root_dir))

In [5]:
from src.data_loader import load_data_tidy
from src.models import build_custom_processor, apply_log_transform, split_into_train_dev_test, train_and_evaluate, engineer_date_features, train_A_test_B, k_fold, exp4_train_AkB_test_remaining_B

In [7]:
df_4_6 = load_data_tidy("dataset_4_6")
df_7_9 = load_data_tidy("dataset_7_9")

In [8]:
print(df_4_6.head())


   Order date  CLASSIFY_CD  CUST_CD BRAND_CD SUPPLIER_CD  \
0  2022-04-04     21034701       74     HAK1        8107   
1  2022-04-26     23028764     4180     S3M1        8484   
2  2022-04-19     21059031       74     THT1        8233   
3  2022-04-22      3731227       74     MSM1        3013   
4  2022-04-22      3954111       74     MSM1        3081   

   Sales order line number  Consider count hodiday Saturday  SO QTY  \
0                        1                                4       1   
1                        1                                1      10   
2                        1                                4       1   
3                        1                                0       1   
4                        1                                0       2   

  PACKING RANK  PRODUCT ATTRIBUTION  ...  DIRECT SHIP FLG  DELI_DIV  label  \
0            A                    1  ...                0        00      1   
1            Z                    1  ...                

In [9]:
print(df_7_9.head())

   Order date  CLASSIFY_CD  CUST_CD BRAND_CD SUPPLIER_CD  \
0  2022-08-25     21031103   344472     OSA1        8121   
1  2022-08-12      3712333      641     MSM1        0263   
2  2022-07-08     21045918    32167     SMC1        9102   
3  2022-07-08     21058302   710046     MIB1        9163   
4  2022-08-25     21050569   107539     NIQ1        9176   

   Sales order line number  Consider count hodiday Saturday  SO QTY  \
0                        8                                4       1   
1                        1                                0       2   
2                        1                                4       1   
3                        1                                4       2   
4                        6                                4       2   

  PACKING RANK  PRODUCT ATTRIBUTION  ...  DIRECT SHIP FLG  DELI_DIV  label  \
0            Z                    1  ...                0        00      1   
1            A                    1  ...                

### SPLIT INTO TRAIN - DEV - TEST

In [10]:
target_col = ['label']
date_cols = ['Order date', 'VSD']

In [11]:
df_4_6 = engineer_date_features(df_4_6, date_cols)
df_7_9 = engineer_date_features(df_7_9, date_cols)

Datetime processing complete: Created 5 new feature columns.
Datetime processing complete: Created 5 new feature columns.


In [12]:
df_4_6.columns

Index(['CLASSIFY_CD', 'CUST_CD', 'BRAND_CD', 'SUPPLIER_CD',
       'Sales order line number', 'Consider count hodiday Saturday', 'SO QTY',
       'PACKING RANK', 'PRODUCT ATTRIBUTION', 'SPECIAL DIV', 'LOGICAL PLANT',
       'PURCHASE AMOUNT', 'DIRECT SHIP FLG', 'DELI_DIV', 'label', 'Ship Mode',
       'PACK QTY', 'WEIGHT PER PIECE', 'SUPPLIER_DIV', 'SO_DAY_OF_MONTH',
       'SO_DAY_OF_WEEK', 'SO_TIME', 'Expected_Lead_Time', 'Order_Month',
       'Order_DayOfWeek', 'Is_Weekend_Order', 'VSD_Month'],
      dtype='object')

In [13]:
num_cols = [
                'SO QTY',
                'PURCHASE AMOUNT',
                'PACK QTY',
                'WEIGHT PER PIECE',
                'SO_DAY_OF_MONTH',
                'SO_TIME',
                'Expected_Lead_Time'
]
cat_cols= df_4_6.columns.difference(num_cols).difference(target_col).difference(date_cols).tolist()

In [14]:
target_encode_cols = ['BRAND_CD', 'CUST_CD', 'SUPPLIER_CD', 'CLASSIFY_CD', 'Sales order line number']
onehot_encode_cols = ['Order_Month', 'VSD_Month', 'Order_DayOfWeek', 'Is_Weekend_Order', 'Consider count hodiday Saturday', 'DELI_DIV', 'DIRECT SHIP FLG', 'LOGICAL PLANT', 'PACKING RANK', 'PRODUCT ATTRIBUTION', 'SO_DAY_OF_WEEK', 'SPECIAL DIV', 'SUPPLIER_DIV', 'Ship Mode']

In [15]:
apply_log_transform_cols = ['SO QTY', 'PURCHASE AMOUNT', 'PACK QTY', 'WEIGHT PER PIECE', 'Expected_Lead_Time']
df_4_6 =apply_log_transform(df_4_6, apply_log_transform_cols)
apply_log_transform_cols_7_9 = ['SO QTY', 'PURCHASE AMOUNT', 'PACK QTY', 'WEIGHT PER PIECE', 'Expected_Lead_Time']
df_7_9 = apply_log_transform(df_7_9, apply_log_transform_cols_7_9)

Successfully applied Log Transformation to 5 columns.
Successfully applied Log Transformation to 5 columns.


In [18]:
X_A = df_4_6.drop(columns=target_col)
y_A = df_4_6[target_col]
X_B = df_7_9.drop(columns=target_col)
y_B = df_7_9[target_col]

In [19]:
X_train_A, X_dev_A, X_test_A, y_train_A, y_dev_A, y_test_A = split_into_train_dev_test(df_4_6, target_col[0])
X_train_B, X_dev_B, X_test_B, y_train_B, y_dev_B, y_test_B = split_into_train_dev_test(df_7_9, target_col[0])

In [20]:
print("Shapes of datasets:")
print(f"X_A: {X_A.shape}, y_A: {y_A.shape}")
print(f"X_train_A: {X_train_A.shape}, y_train_A: {y_train_A.shape}")
print(f"X_dev_A: {X_dev_A.shape}, y_dev_A: {y_dev_A.shape}")
print(f"X_test_A: {X_test_A.shape}, y_test_A: {y_test_A.shape}")    

Shapes of datasets:
X_A: (399053, 26), y_A: (399053, 1)
X_train_A: (319242, 26), y_train_A: (319242,)
X_dev_A: (39905, 26), y_dev_A: (39905,)
X_test_A: (39906, 26), y_test_A: (39906,)


In [21]:
print("Shapes of datasets:")
print(f"X_B: {X_B.shape}, y_B: {y_B.shape}")
print(f"X_train_B: {X_train_B.shape}, y_train_B: {y_train_B.shape}")
print(f"X_dev_B: {X_dev_B.shape}, y_dev_B: {y_dev_B.shape}")
print(f"X_test_B: {X_test_B.shape}, y_test_B: {y_test_B.shape}")    

Shapes of datasets:
X_B: (1074897, 26), y_B: (1074897, 1)
X_train_B: (859917, 26), y_train_B: (859917,)
X_dev_B: (107490, 26), y_dev_B: (107490,)
X_test_B: (107490, 26), y_test_B: (107490,)


In [22]:
processor = build_custom_processor(target_encode_cols, onehot_encode_cols, num_cols)

## KNN model

In [17]:
knn_model = KNeighborsClassifier(n_neighbors=5, n_jobs=-1) 

trained_knn_pipeline_A = train_and_evaluate(
    model=knn_model, 
    X_train=X_train_A, y_train=y_train_A, 
    X_dev=X_dev_A,     y_dev=y_dev_A, 
    X_test=X_test_A,   y_test=y_test_A, 
    preprocessor=processor
)

Training model: KNeighborsClassifier...

DEV SET REPORT (VALIDATION) 
 Used for monitoring Overfitting and Hyperparameter Tuning
              precision    recall  f1-score   support

           0     0.9934    0.9402    0.9661     38932
           1     0.2388    0.7503    0.3623       973

    accuracy                         0.9356     39905
   macro avg     0.6161    0.8452    0.6642     39905
weighted avg     0.9750    0.9356    0.9514     39905

[+] AUC-ROC (Dev): 0.8908

 TEST SET REPORT (FINAL RESULT) 
 Final performance metrics for project reporting
[-] Confusion Matrix (Test):
[[36700  2232]
 [  213   761]]

[-] Classification Report (Test):
              precision    recall  f1-score   support

           0     0.9942    0.9427    0.9678     38932
           1     0.2543    0.7813    0.3837       974

    accuracy                         0.9387     39906
   macro avg     0.6242    0.8620    0.6757     39906
weighted avg     0.9762    0.9387    0.9535     39906

[+] AUC-ROC (

In [18]:
knn_model = KNeighborsClassifier(n_neighbors=5, n_jobs=-1) 

trained_knn_pipeline_B = train_and_evaluate(
    model=knn_model, 
    X_train=X_train_B, y_train=y_train_B, 
    X_dev=X_dev_B,     y_dev=y_dev_B, 
    X_test=X_test_B,   y_test=y_test_B, 
    preprocessor=processor
)

Training model: KNeighborsClassifier...

DEV SET REPORT (VALIDATION) 
 Used for monitoring Overfitting and Hyperparameter Tuning
              precision    recall  f1-score   support

           0     0.9952    0.9591    0.9768    104868
           1     0.3325    0.8139    0.4721      2622

    accuracy                         0.9556    107490
   macro avg     0.6638    0.8865    0.7245    107490
weighted avg     0.9790    0.9556    0.9645    107490

[+] AUC-ROC (Dev): 0.9204

 TEST SET REPORT (FINAL RESULT) 
 Final performance metrics for project reporting
[-] Confusion Matrix (Test):
[[100573   4296]
 [   519   2102]]

[-] Classification Report (Test):
              precision    recall  f1-score   support

           0     0.9949    0.9590    0.9766    104869
           1     0.3285    0.8020    0.4661      2621

    accuracy                         0.9552    107490
   macro avg     0.6617    0.8805    0.7214    107490
weighted avg     0.9786    0.9552    0.9642    107490

[+] AUC-R

## LIGHTGBM MODEL

In [24]:
lightgbm_model = LGBMClassifier(
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=31,
    random_state=42,     
    n_jobs=-1,
    verbose=-1,
)

train_lightgbm_A = train_and_evaluate(
    model=lightgbm_model, 
    X_train=X_train_A, y_train=y_train_A, 
    X_dev=X_dev_A,     y_dev=y_dev_A, 
    X_test=X_test_A,   y_test=y_test_A, 
    preprocessor=processor
)

Training model: LGBMClassifier...

DEV SET REPORT (VALIDATION) 
 Used for monitoring Overfitting and Hyperparameter Tuning


/home/huyy-giaa/miniconda3/envs/DS108_TH/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


              precision    recall  f1-score   support

           0     0.9968    0.8782    0.9337     38932
           1     0.1538    0.8859    0.2621       973

    accuracy                         0.8784     39905
   macro avg     0.5753    0.8821    0.5979     39905
weighted avg     0.9762    0.8784    0.9174     39905



/home/huyy-giaa/miniconda3/envs/DS108_TH/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[+] AUC-ROC (Dev): 0.9530

 TEST SET REPORT (FINAL RESULT) 
 Final performance metrics for project reporting


/home/huyy-giaa/miniconda3/envs/DS108_TH/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Confusion Matrix (Test):
[[34055  4877]
 [   97   877]]

Classification Report (Test):
              precision    recall  f1-score   support

           0     0.9972    0.8747    0.9319     38932
           1     0.1524    0.9004    0.2607       974

    accuracy                         0.8754     39906
   macro avg     0.5748    0.8876    0.5963     39906
weighted avg     0.9765    0.8754    0.9156     39906

AUC-ROC (Test): 0.9569


/home/huyy-giaa/miniconda3/envs/DS108_TH/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [25]:
train_lightgbm_B = train_and_evaluate(
    model=lightgbm_model, 
    X_train=X_train_B, y_train=y_train_B, 
    X_dev=X_dev_B,     y_dev=y_dev_B, 
    X_test=X_test_B,   y_test=y_test_B, 
    preprocessor=processor
)

Training model: LGBMClassifier...

DEV SET REPORT (VALIDATION) 
 Used for monitoring Overfitting and Hyperparameter Tuning


/home/huyy-giaa/miniconda3/envs/DS108_TH/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


              precision    recall  f1-score   support

           0     0.9954    0.8914    0.9405    104868
           1     0.1613    0.8352    0.2703      2622

    accuracy                         0.8900    107490
   macro avg     0.5783    0.8633    0.6054    107490
weighted avg     0.9751    0.8900    0.9242    107490



/home/huyy-giaa/miniconda3/envs/DS108_TH/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[+] AUC-ROC (Dev): 0.9413

 TEST SET REPORT (FINAL RESULT) 
 Final performance metrics for project reporting


/home/huyy-giaa/miniconda3/envs/DS108_TH/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Confusion Matrix (Test):
[[93537 11332]
 [  406  2215]]

Classification Report (Test):
              precision    recall  f1-score   support

           0     0.9957    0.8919    0.9410    104869
           1     0.1635    0.8451    0.2740      2621

    accuracy                         0.8908    107490
   macro avg     0.5796    0.8685    0.6075    107490
weighted avg     0.9754    0.8908    0.9247    107490



/home/huyy-giaa/miniconda3/envs/DS108_TH/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


AUC-ROC (Test): 0.9452


## CATBOOST MODEL

In [26]:
cat_model = CatBoostClassifier(
    iterations=500,          
    learning_rate=0.05,
    depth=6,                 
    subsample=0.8,
    colsample_bylevel=0.8,   
    thread_count=-1,         
    verbose=False,           
    random_state=42
)

In [27]:
train_cat_A = train_and_evaluate(
    model=cat_model, 
    X_train=X_train_A, y_train=y_train_A, 
    X_dev=X_dev_A,     y_dev=y_dev_A, 
    X_test=X_test_A,   y_test=y_test_A, 
    preprocessor=processor
)

Training model: CatBoostClassifier...

DEV SET REPORT (VALIDATION) 
 Used for monitoring Overfitting and Hyperparameter Tuning
              precision    recall  f1-score   support

           0     0.9951    0.9292    0.9610     38932
           1     0.2237    0.8160    0.3511       973

    accuracy                         0.9265     39905
   macro avg     0.6094    0.8726    0.6561     39905
weighted avg     0.9763    0.9265    0.9461     39905

[+] AUC-ROC (Dev): 0.9475

 TEST SET REPORT (FINAL RESULT) 
 Final performance metrics for project reporting
Confusion Matrix (Test):
[[36144  2788]
 [  152   822]]

Classification Report (Test):
              precision    recall  f1-score   support

           0     0.9958    0.9284    0.9609     38932
           1     0.2277    0.8439    0.3586       974

    accuracy                         0.9263     39906
   macro avg     0.6118    0.8862    0.6598     39906
weighted avg     0.9771    0.9263    0.9462     39906

AUC-ROC (Test): 0.9537


In [28]:
train_cat_B = train_and_evaluate(
    model=cat_model, 
    X_train=X_train_B, y_train=y_train_B, 
    X_dev=X_dev_B,     y_dev=y_dev_B, 
    X_test=X_test_B,   y_test=y_test_B, 
    preprocessor=processor
)

Training model: CatBoostClassifier...

DEV SET REPORT (VALIDATION) 
 Used for monitoring Overfitting and Hyperparameter Tuning
              precision    recall  f1-score   support

           0     0.9961    0.6323    0.7736    104868
           1     0.0577    0.9012    0.1085      2622

    accuracy                         0.6389    107490
   macro avg     0.5269    0.7668    0.4410    107490
weighted avg     0.9732    0.6389    0.7573    107490

[+] AUC-ROC (Dev): 0.9050

 TEST SET REPORT (FINAL RESULT) 
 Final performance metrics for project reporting
Confusion Matrix (Test):
[[66154 38715]
 [  238  2383]]

Classification Report (Test):
              precision    recall  f1-score   support

           0     0.9964    0.6308    0.7726    104869
           1     0.0580    0.9092    0.1090      2621

    accuracy                         0.6376    107490
   macro avg     0.5272    0.7700    0.4408    107490
weighted avg     0.9735    0.6376    0.7564    107490

AUC-ROC (Test): 0.9096


## XGBOOST MODEL

In [29]:
xgb_model = XGBClassifier(
    n_estimators=500,        
    learning_rate=0.05,      
    max_depth=6,             
    subsample=0.8,           
    colsample_bytree=0.8,    
    tree_method='hist',      
    n_jobs=-1,               
    random_state=42
)

In [30]:
train_xgb_A = train_and_evaluate(
    model=xgb_model, 
    X_train=X_train_A, y_train=y_train_A, 
    X_dev=X_dev_A,     y_dev=y_dev_A, 
    X_test=X_test_A,   y_test=y_test_A, 
    preprocessor=processor
)

Training model: XGBClassifier...

DEV SET REPORT (VALIDATION) 
 Used for monitoring Overfitting and Hyperparameter Tuning
              precision    recall  f1-score   support

           0     0.9949    0.9374    0.9653     38932
           1     0.2439    0.8078    0.3747       973

    accuracy                         0.9343     39905
   macro avg     0.6194    0.8726    0.6700     39905
weighted avg     0.9766    0.9343    0.9509     39905

[+] AUC-ROC (Dev): 0.9499

 TEST SET REPORT (FINAL RESULT) 
 Final performance metrics for project reporting
Confusion Matrix (Test):
[[36492  2440]
 [  164   810]]

Classification Report (Test):
              precision    recall  f1-score   support

           0     0.9955    0.9373    0.9656     38932
           1     0.2492    0.8316    0.3835       974

    accuracy                         0.9347     39906
   macro avg     0.6224    0.8845    0.6745     39906
weighted avg     0.9773    0.9347    0.9513     39906

AUC-ROC (Test): 0.9564


In [31]:
train_xgb_B = train_and_evaluate(
    model=xgb_model, 
    X_train=X_train_B, y_train=y_train_B, 
    X_dev=X_dev_B,     y_dev=y_dev_B, 
    X_test=X_test_B,   y_test=y_test_B, 
    preprocessor=processor
)

Training model: XGBClassifier...

DEV SET REPORT (VALIDATION) 
 Used for monitoring Overfitting and Hyperparameter Tuning
              precision    recall  f1-score   support

           0     0.9962    0.8714    0.9296    104868
           1     0.1442    0.8665    0.2473      2622

    accuracy                         0.8713    107490
   macro avg     0.5702    0.8690    0.5885    107490
weighted avg     0.9754    0.8713    0.9130    107490

[+] AUC-ROC (Dev): 0.9412

 TEST SET REPORT (FINAL RESULT) 
 Final performance metrics for project reporting
Confusion Matrix (Test):
[[91391 13478]
 [  343  2278]]

Classification Report (Test):
              precision    recall  f1-score   support

           0     0.9963    0.8715    0.9297    104869
           1     0.1446    0.8691    0.2479      2621

    accuracy                         0.8714    107490
   macro avg     0.5704    0.8703    0.5888    107490
weighted avg     0.9755    0.8714    0.9131    107490

AUC-ROC (Test): 0.9440


### TRAIN A TEST B

In [32]:
X_A_train = df_4_6.drop(columns=['label'])
y_A_train = df_4_6['label']
X_B_test = df_7_9.drop(columns=['label'])
y_B_test = df_7_9['label']

In [35]:
print("Nhãn của tập A:", y_A_train.unique())
print("Nhãn của tập B:", y_B_test.unique())

Nhãn của tập A: [1 0]
Nhãn của tập B: [1 0]


In [36]:
train_lightgbm_A_B = train_A_test_B(
    model = lightgbm_model,
    X_train = X_A_train,
    y_train = y_A_train.astype(int),
    X_test = X_B_test,
    y_test = y_B_test.astype(int),
    preprocessor=processor
)

Training model: LGBMClassifier...

 TEST SET REPORT (FINAL RESULT) 
 Final performance metrics for project reporting


/home/huyy-giaa/miniconda3/envs/DS108_TH/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Confusion Matrix (Test)
[[1006751   41929]
 [  21307    4910]]

Classification Report (Test):
              precision    recall  f1-score   support

           0     0.9793    0.9600    0.9696   1048680
           1     0.1048    0.1873    0.1344     26217

    accuracy                         0.9412   1074897
   macro avg     0.5421    0.5737    0.5520   1074897
weighted avg     0.9579    0.9412    0.9492   1074897



/home/huyy-giaa/miniconda3/envs/DS108_TH/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


AUC-ROC (Test): 0.7171


### TRAIN B TEST A

In [37]:
X_B_train = df_7_9.drop(columns=['label'])
y_B_train = df_7_9['label']
X_A_test = df_4_6.drop(columns=['label'])
y_A_test = df_4_6['label']

In [39]:
train_lightgbm_B_A = train_A_test_B(
    model = lightgbm_model,
    X_train = X_B_train,
    y_train = y_B_train,
    X_test = X_A_test,
    y_test = y_A_test,
    preprocessor=processor
)

Training model: LGBMClassifier...

 TEST SET REPORT (FINAL RESULT) 
 Final performance metrics for project reporting


/home/huyy-giaa/miniconda3/envs/DS108_TH/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Confusion Matrix (Test)
[[309080  80240]
 [  5300   4433]]

Classification Report (Test):
              precision    recall  f1-score   support

           0     0.9831    0.7939    0.8784    389320
           1     0.0524    0.4555    0.0939      9733

    accuracy                         0.7856    399053
   macro avg     0.5177    0.6247    0.4862    399053
weighted avg     0.9604    0.7856    0.8593    399053



/home/huyy-giaa/miniconda3/envs/DS108_TH/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


AUC-ROC (Test): 0.7094


### K-FOLD (A + B) K=5

In [40]:
train_lightgbm_k_fold = k_fold(
    model=lightgbm_model,
    X_A = X_A,
    y_A = y_A,
    X_B = X_B, 
    y_B = y_B,
    preprocessor=processor
)

Concat Dataset: 1473950 samples | Class ratio: {(0,): 0.975609756097561, (1,): 0.024390243902439025}

==================== FOLD 1/5 ====================
Training model: LGBMClassifier...


/home/huyy-giaa/miniconda3/envs/DS108_TH/lib/python3.10/site-packages/sklearn/preprocessing/_label.py:110: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/home/huyy-giaa/miniconda3/envs/DS108_TH/lib/python3.10/site-packages/sklearn/preprocessing/_label.py:93: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/home/huyy-giaa/miniconda3/envs/DS108_TH/lib/python3.10/site-packages/sklearn/preprocessing/_label.py:129: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, dtype=self.classes_.dtype, warn=True)
/home/huyy-giaa/miniconda3/envs/DS108_TH/lib/python3.10/site-packages/sklearn/utils/va

Confusion Matrix (Fold 1):
[[244087  43513]
 [   629   6561]]

Classification Report (Fold 1):
              precision    recall  f1-score   support

           0     0.9974    0.8487    0.9171    287600
           1     0.1310    0.9125    0.2291      7190

    accuracy                         0.8503    294790
   macro avg     0.5642    0.8806    0.5731    294790
weighted avg     0.9763    0.8503    0.9003    294790



/home/huyy-giaa/miniconda3/envs/DS108_TH/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


AUC-ROC (Fold 1): 0.9569

==================== FOLD 2/5 ====================
Training model: LGBMClassifier...


...

AUC-ROC (Fold 4): 0.9437

==================== FOLD 5/5 ====================
Training model: LGBMClassifier...


/home/huyy-giaa/miniconda3/envs/DS108_TH/lib/python3.10/site-packages/sklearn/preprocessing/_label.py:110: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/home/huyy-giaa/miniconda3/envs/DS108_TH/lib/python3.10/site-packages/sklearn/preprocessing/_label.py:93: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/home/huyy-giaa/miniconda3/envs/DS108_TH/lib/python3.10/site-packages/sklearn/preprocessing/_label.py:129: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, dtype=self.classes_.dtype, warn=True)
/home/huyy-giaa/miniconda3/envs/DS108_TH/lib/python3.10/site-packages/sklearn/utils/va

Confusion Matrix (Fold 5):
[[217053  70547]
 [   613   6577]]

Classification Report (Fold 5):
              precision    recall  f1-score   support

           0     0.9972    0.7547    0.8592    287600
           1     0.0853    0.9147    0.1560      7190

    accuracy                         0.7586    294790
   macro avg     0.5412    0.8347    0.5076    294790
weighted avg     0.9749    0.7586    0.8420    294790



/home/huyy-giaa/miniconda3/envs/DS108_TH/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


AUC-ROC (Fold 5): 0.9358

Conclusion 5-FOLD CROSS VALIDATION
Macro-F1  | Per fold: ['0.5731', '0.5674', '0.5918', '0.5427', '0.5076']
           Avg: 0.5565 ± 0.0291
AUC-ROC   | Per fold: ['0.9569', '0.9498', '0.9599', '0.9437', '0.9358']
           Avg: 0.9492 ± 0.0088


### TRAIN: (A + B) TEST: (1 - K).B

In [41]:
train_lightgbm_exp4 = exp4_train_AkB_test_remaining_B(
    model=lightgbm_model,
    X_A = X_A,
    y_A = y_A,
    X_B = X_B, 
    y_B = y_B,
    preprocessor=processor
)


  K = 10% | Train: A + 10%B  →  Test: 90%B
  Train size : 506,542 samples  (A=399,053 + 10%B=107,489)
  Test size  : 967,408 samples (90%B)

  Training LGBMClassifier...


/home/huyy-giaa/miniconda3/envs/DS108_TH/lib/python3.10/site-packages/sklearn/preprocessing/_label.py:110: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/home/huyy-giaa/miniconda3/envs/DS108_TH/lib/python3.10/site-packages/sklearn/preprocessing/_label.py:93: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/home/huyy-giaa/miniconda3/envs/DS108_TH/lib/python3.10/site-packages/sklearn/preprocessing/_label.py:129: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, dtype=self.classes_.dtype, warn=True)
/home/huyy-giaa/miniconda3/envs/DS108_TH/lib/python3.10/site-packages/sklearn/utils/va


  Confusion Matrix:
[[814732 129081]
 [  4101  19494]]

  Classification Report:
              precision    recall  f1-score   support

           0     0.9950    0.8632    0.9244    943813
           1     0.1312    0.8262    0.2265     23595

    accuracy                         0.8623    967408
   macro avg     0.5631    0.8447    0.5754    967408
weighted avg     0.9739    0.8623    0.9074    967408



/home/huyy-giaa/miniconda3/envs/DS108_TH/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


  AUC-ROC: 0.9289

  K = 30% | Train: A + 30%B  →  Test: 70%B
  Train size : 721,522 samples  (A=399,053 + 30%B=322,469)
  Test size  : 752,428 samples (70%B)

  Training LGBMClassifier...


...

  AUC-ROC: 0.9546

  K = 90% | Train: A + 90%B  →  Test: 9%B
  Train size : 1,366,460 samples  (A=399,053 + 90%B=967,407)
  Test size  : 107,490 samples (9%B)

  Training LGBMClassifier...


/home/huyy-giaa/miniconda3/envs/DS108_TH/lib/python3.10/site-packages/sklearn/preprocessing/_label.py:110: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/home/huyy-giaa/miniconda3/envs/DS108_TH/lib/python3.10/site-packages/sklearn/preprocessing/_label.py:93: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/home/huyy-giaa/miniconda3/envs/DS108_TH/lib/python3.10/site-packages/sklearn/preprocessing/_label.py:129: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, dtype=self.classes_.dtype, warn=True)
/home/huyy-giaa/miniconda3/envs/DS108_TH/lib/python3.10/site-packages/sklearn/utils/va


  Confusion Matrix:
[[75102 29766]
 [  227  2395]]

  Classification Report:
              precision    recall  f1-score   support

           0     0.9970    0.7162    0.8336    104868
           1     0.0745    0.9134    0.1377      2622

    accuracy                         0.7210    107490
   macro avg     0.5357    0.8148    0.4856    107490
weighted avg     0.9745    0.7210    0.8166    107490



/home/huyy-giaa/miniconda3/envs/DS108_TH/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


  AUC-ROC: 0.9286

  EXP4 SUMMARY — LGBMClassifier
     K% |   Train size |  Test size |   Macro-F1 |      AUC
  -------------------------------------------------------
    10% |      506,542 |    967,408 |     0.5754 |   0.9289
    30% |      721,522 |    752,428 |     0.5812 |   0.9491
    50% |      936,501 |    537,449 |     0.5737 |   0.9470
    70% |    1,151,480 |    322,470 |     0.5891 |   0.9546
    90% |    1,366,460 |    107,490 |     0.4856 |   0.9286
